# Logistic Regression Experiment — IEEE-CIS Fraud Detection

## 0. Setup & Imports

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'dagshub', 'optuna', '--quiet'], capture_output=True)

import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn, dagshub, optuna
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.base import BaseEstimator, TransformerMixin
print('Ready!')

In [ ]:
DAGSHUB_USERNAME = 'YOUR_DAGSHUB_USERNAME'
DAGSHUB_REPO     = 'YOUR_REPO_NAME'
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment('LogisticRegression_Training')

## 1. Data Loading

In [ ]:
BASE = '/kaggle/input/ieee-fraud-detection/'
train = pd.read_csv(BASE+'train_transaction.csv').merge(pd.read_csv(BASE+'train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(BASE+'test_transaction.csv').merge(pd.read_csv(BASE+'test_identity.csv'),  on='TransactionID', how='left')
print(train.shape, test.shape)

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='LR_Cleaning'):
    # Logistic Regression is sensitive to missing and scale — aggressive cleaning
    drop_cols = train.isnull().mean()[lambda x: x > 0.5].index.tolist()  # stricter: 50%
    train.drop(columns=drop_cols, inplace=True)
    test.drop(columns=[c for c in drop_cols if c in test], inplace=True)

    const_cols = [c for c in train.columns if train[c].nunique(dropna=False) <= 1]
    train.drop(columns=const_cols, inplace=True)
    test.drop(columns=[c for c in const_cols if c in test], inplace=True)

    for col in ['P_emaildomain','R_emaildomain']:
        if col in train:
            top = train[col].value_counts().nlargest(10).index
            train[col] = train[col].where(train[col].isin(top), 'other')
            test[col]  = test[col].where(test[col].isin(top), 'other')

    mlflow.log_param('missing_threshold', 0.5)
    mlflow.log_param('dropped_high_missing', len(drop_cols))
    mlflow.log_metric('cols_remaining', train.shape[1])
    print(f'After cleaning: {train.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='LR_Feature_Engineering'):

    def engineer(df):
        df = df.copy()
        df['hour']        = (df['TransactionDT'] / 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
        # Log transform for skewed amounts (important for LR)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_sqrt']  = np.sqrt(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        if 'P_emaildomain' in df and 'R_emaildomain' in df:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        df['nan_count'] = df.isnull().sum(axis=1)
        return df

    train = engineer(train)
    test  = engineer(test)

    TARGET   = 'isFraud'
    DROP_COLS= ['TransactionID', 'TransactionDT', TARGET]

    cat_cols = [c for c in train.select_dtypes(include='object').columns if c not in DROP_COLS]
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]]).astype(str)
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))

    feature_cols = [c for c in train.columns if c not in DROP_COLS]
    X = train[feature_cols].fillna(train[feature_cols].median())
    y = train[TARGET]
    X_test = test[feature_cols].fillna(train[feature_cols].median())

    mlflow.log_metric('features_after_fe', X.shape[1])
    print(f'Shape: {X.shape}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='LR_Feature_Selection'):

    # Method 1: SelectKBest with f_classif (ANOVA F-score)
    selector_kbest = SelectKBest(f_classif, k=80)
    X_kbest = selector_kbest.fit_transform(X, y)
    kbest_features = [feature_cols[i] for i in selector_kbest.get_support(indices=True)]

    # Method 2: L1 regularization based selection (Lasso)
    from sklearn.linear_model import Lasso
    scaler_tmp = StandardScaler()
    X_scaled_tmp = scaler_tmp.fit_transform(X)
    lasso = LogisticRegression(C=0.01, penalty='l1', solver='liblinear', max_iter=1000)
    lasso.fit(X_scaled_tmp, y)
    lasso_mask = lasso.coef_[0] != 0
    lasso_features = [f for f, m in zip(feature_cols, lasso_mask) if m]

    # Combine: union of both methods
    selected = list(set(kbest_features) | set(lasso_features))

    mlflow.log_param('fs_method_1', 'SelectKBest_f_classif_k80')
    mlflow.log_param('fs_method_2', 'L1_Lasso_C001')
    mlflow.log_metric('kbest_selected', len(kbest_features))
    mlflow.log_metric('lasso_selected', len(lasso_features))
    mlflow.log_metric('union_selected', len(selected))

    X_sel      = X[selected]
    X_test_sel = X_test[selected]
    print(f'KBest: {len(kbest_features)} | Lasso: {len(lasso_features)} | Union: {len(selected)}')

## 5. Training

### 5a. Underfitted — Very Strong Regularization

In [ ]:
with mlflow.start_run(run_name='LR_Underfitted'):
    params_u = dict(C=0.0001, penalty='l2', solver='lbfgs', max_iter=100)
    scaler_u = StandardScaler()
    X_scaled = scaler_u.fit_transform(X_sel)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(LogisticRegression(**params_u), X_scaled, y, cv=cv, scoring='roc_auc')
    mlflow.log_params(params_u)
    mlflow.log_metric('cv_auc_mean', scores.mean())
    mlflow.log_param('note', 'underfitted_very_high_regularization')
    print(f'[UNDERFITTED] CV AUC: {scores.mean():.4f} — high regularization suppresses all signal')

### 5b. Overfitted — No Regularization + Many Iterations

In [ ]:
with mlflow.start_run(run_name='LR_Overfitted'):
    # LR doesn't overfit as dramatically as tree models, but we can show the concept
    params_o = dict(C=1000, penalty='l2', solver='lbfgs', max_iter=5000)
    scaler_o = StandardScaler()
    X_scaled_o = scaler_o.fit_transform(X_sel)
    m_o = LogisticRegression(**params_o)
    m_o.fit(X_scaled_o, y)
    train_auc = roc_auc_score(y, m_o.predict_proba(X_scaled_o)[:, 1])
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_auc = cross_val_score(LogisticRegression(**params_o), X_scaled_o, y,
                              cv=cv, scoring='roc_auc').mean()
    mlflow.log_params(params_o)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('cv_auc', cv_auc)
    mlflow.log_metric('overfit_gap', train_auc - cv_auc)
    mlflow.log_param('note', 'very_low_regularization_C1000')
    print(f'[NEAR-OVERFIT] Train: {train_auc:.4f} CV: {cv_auc:.4f}')

### 5c. Optuna Tuning

In [ ]:
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X_sel)

def lr_objective(trial):
    params = {
        'C':        trial.suggest_float('C', 1e-3, 10.0, log=True),
        'penalty':  trial.suggest_categorical('penalty', ['l1', 'l2']),
        'solver':   'liblinear',
        'max_iter': 1000
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(LogisticRegression(**params), X_scaled_final, y,
                           cv=cv, scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize')
study.optimize(lr_objective, n_trials=20)
best_params = study.best_params
best_params['solver'] = 'liblinear'
best_params['max_iter'] = 1000
print(f'Best AUC: {study.best_value:.4f} | Params: {best_params}')

### 5d. Final CV + Pipeline

In [ ]:
with mlflow.start_run(run_name='LR_Final_CV') as final_run:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_sel))
    fold_aucs = []
    scaler_cv = StandardScaler()
    X_sc = scaler_cv.fit_transform(X_sel)
    X_test_sc = scaler_cv.transform(X_test_sel)

    for fold, (tr_i, val_i) in enumerate(cv.split(X_sc, y)):
        m = LogisticRegression(**best_params)
        m.fit(X_sc[tr_i], y.iloc[tr_i])
        val_pred = m.predict_proba(X_sc[val_i])[:, 1]
        oof[val_i] = val_pred
        test_preds += m.predict_proba(X_test_sc)[:, 1] / 5
        fa = roc_auc_score(y.iloc[val_i], val_pred)
        fold_aucs.append(fa)
        print(f'  Fold {fold+1}: {fa:.4f}')

    oof_auc = roc_auc_score(y, oof)
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('cv_auc_mean', np.mean(fold_aucs))
    print(f'OOF AUC: {oof_auc:.4f}')


class LRPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, selected_features=None):
        self.selected_features = selected_features
        self.label_encoders_ = {}
        self.cat_cols_ = []
        self.medians_ = None

    def fit(self, X, y=None):
        df = self._engineer(X.copy())
        self.cat_cols_ = df.select_dtypes(include='object').columns.tolist()
        for col in self.cat_cols_:
            le = LabelEncoder(); le.fit(df[col].astype(str))
            self.label_encoders_[col] = le
        if self.selected_features:
            df = df[[f for f in self.selected_features if f in df.columns]]
        self.medians_ = df.median()
        return self

    def transform(self, X):
        df = self._engineer(X.copy())
        for col in self.cat_cols_:
            if col in df:
                le = self.label_encoders_[col]
                df[col] = df[col].astype(str).map(lambda x: x if x in le.classes_ else le.classes_[0])
                df[col] = le.transform(df[col])
        if self.selected_features:
            df = df[[f for f in self.selected_features if f in df.columns]]
        return df.fillna(self.medians_)

    def _engineer(self, df):
        df['hour']       = (df['TransactionDT'] / 3600) % 24
        df['day_of_week']= (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']   = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_sqrt']  = np.sqrt(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['nan_count'] = df.isnull().sum(axis=1)
        if 'P_emaildomain' in df and 'R_emaildomain' in df:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        return df


X_raw = train.drop(columns=['isFraud','TransactionID'], errors='ignore')
y_raw = train['isFraud']

lr_pipeline = Pipeline([
    ('preprocessor', LRPreprocessor(selected_features=selected)),
    ('scaler',       StandardScaler()),
    ('classifier',   LogisticRegression(**best_params))
])
lr_pipeline.fit(X_raw, y_raw)

with mlflow.start_run(run_name='LR_Pipeline_Registry'):
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.sklearn.log_model(
        sk_model=lr_pipeline,
        artifact_path='lr_fraud_pipeline',
        registered_model_name='LogisticRegression_Fraud_Pipeline'
    )
    print('LR pipeline registered!')

np.save('lr_test_preds.npy', test_preds)